# MultiSceneDatasetV4 + AssetPreloadManagerV2 + TrainSchedulerV6 集成测试与教学演示

本 notebook 仿照 `StreetForward_Asset_System_Integration_Demo.ipynb` 的资产流转方式，覆盖：

1. **资产导出命令**：通过 CLI（`build_streetforward_scene_assets.py` / `build_streetforward_segment_assets.py`）导出 scene/segment 资产。
2. **资产导入验证**：通过 `StreetForwardAssetStore` 校验资产目录、manifest 与 `image_table` 元信息。
3. **batch API 集成断言**：验证 `MultiSceneDatasetV4.get_segment_batch_from_image_refs()` 的关键契约。
4. **预热与调度模拟训练**：演示 preload hint、`TrainSchedulerV6` 取 batch 与 toy 训练循环。

> 建议在 `conda drivestudio-new` 环境中运行，并设置：
>
> `PYTHONPATH=/root/drivestudio-coding`


In [5]:
# ====== 0) 环境准备 ======

import os
import sys
import time
import subprocess
from pathlib import Path


PROJECT_ROOT = Path('/root/drivestudio-coding')
os.environ['PYTHONPATH'] = str(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("python:", os.sys.version.split()[0])
print("torch:", torch.__version__)
print("PYTHONPATH:", os.environ.get("PYTHONPATH", "<unset>"))


import numpy as np
import torch
from omegaconf import OmegaConf

from datasets.multi_scene_dataset_v4 import BatchRequestV4
from datasets.streetforward_assets import StreetForwardAssetStore
from tools.train_minimal_streetforward_stage4_3_v6_common import (
    build_multi_scene_dataset_v4,
    build_train_scheduler_v6_from_cfg,
)


def run(cmd: str, check: bool = True):
    """运行 shell 命令并打印输出。"""
    print(f"\n[RUN] {cmd}")
    completed = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    if completed.stdout:
        print(completed.stdout)
    if completed.stderr:
        print(completed.stderr)
    if check and completed.returncode != 0:
        raise RuntimeError(f"Command failed ({completed.returncode}): {cmd}")
    return completed



python: 3.9.23
torch: 2.0.0+cu118
PYTHONPATH: /root/drivestudio-coding
Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [6]:
# ====== 1) 读取完整配置 + 资产导出命令（仿照资产系统集成 notebook） ======

CONFIG_PATH = PROJECT_ROOT / 'configs/minimal_streetforward_stage4_4_one_segment_v6.yaml'
assert CONFIG_PATH.exists(), f'Config not found: {CONFIG_PATH}'

cfg = OmegaConf.load(str(CONFIG_PATH))
assert OmegaConf.select(cfg, 'data') is not None, '配置缺少顶层 data'
assert OmegaConf.select(cfg, 'dataset') is not None, '配置缺少顶层 dataset'
assert OmegaConf.select(cfg, 'data.assets.root') is not None, '配置缺少 data.assets.root'

ASSET_ROOT = Path(OmegaConf.select(cfg, 'data.assets.root'))
DATASET_NAME = str(OmegaConf.select(cfg, 'data.dataset'))
SCENE_ID = int(OmegaConf.select(cfg, 'one_segment.scene_id'))
SEGMENT_ID = int(OmegaConf.select(cfg, 'one_segment.segment_id'))

print('CONFIG_PATH =', CONFIG_PATH)
print('ASSET_ROOT  =', ASSET_ROOT)
print('DATASET     =', DATASET_NAME)
print('SCENE/SEG   =', SCENE_ID, SEGMENT_ID)

scene_cmd = (
    f"cd {PROJECT_ROOT} && "
    f"PYTHONPATH={PROJECT_ROOT} "
    f"python tools/build_streetforward_scene_assets.py "
    f"--config_file {CONFIG_PATH} --scene_id {SCENE_ID}"
)
seg_cmd = (
    f"cd {PROJECT_ROOT} && "
    f"PYTHONPATH={PROJECT_ROOT} "
    f"python tools/build_streetforward_segment_assets.py "
    f"--config_file {CONFIG_PATH} --scene_id {SCENE_ID} --segment_id {SEGMENT_ID}"
)

# 导出可能很慢；默认跳过，资产已存在时推荐保留 False
RUN_EXPORT = True
if RUN_EXPORT:
    run(scene_cmd, check=True)
    run(seg_cmd, check=True)
else:
    print('[SKIP] RUN_EXPORT=False，跳过 CLI 导出（资产已存在时推荐）。')
    print('若尚未导出，请手动执行：')
    print(' ', scene_cmd)
    print(' ', seg_cmd)


CONFIG_PATH = /root/drivestudio-coding/configs/minimal_streetforward_stage4_4_one_segment_v6.yaml
ASSET_ROOT  = /root/autodl-tmp/streetforward_assets
DATASET     = nuscenes
SCENE/SEG   = 1 0

[RUN] cd /root/drivestudio-coding && PYTHONPATH=/root/drivestudio-coding python tools/build_streetforward_scene_assets.py --config_file /root/drivestudio-coding/configs/minimal_streetforward_stage4_4_one_segment_v6.yaml --scene_id 1
[scene-asset] scene_id=1 asset_id=scene-nuscenes-000001-66304760


Loading images: 100%|██████████| 196/196 [00:02<00:00, 76.96it/s]

Loading dynamic masks: 100%|██████████| 196/196 [00:00<00:00, 206.61it/s]

Loading human masks: 100%|██████████| 196/196 [00:00<00:00, 208.91it/s]

Loading vehicle masks: 100%|██████████| 196/196 [00:01<00:00, 192.27it/s]

Loading sky masks: 100%|██████████| 196/196 [00:00<00:00, 413.27it/s]

Loading depth maps for camera 0: 100%|██████████| 196/196 [00:00<00:00, 411.65it/s]

Loading lidar: 100%|██████████| 196/196 [00:01<00:00, 161.29it

In [7]:
# ====== 2) AssetStore 导入检查 + 构建 MultiSceneDatasetV4 ======

scene_pool = ASSET_ROOT / 'scene_pool'
segment_pool = ASSET_ROOT / 'segment_pool'
registries = ASSET_ROOT / 'registries'

print('scene_pool exists   =', scene_pool.exists())
print('segment_pool exists =', segment_pool.exists())
print('registries exists   =', registries.exists())

scene_candidates = sorted([p for p in scene_pool.glob(f'scene-{DATASET_NAME}-{SCENE_ID:06d}-*') if p.is_dir()])
seg_candidates = sorted([
    p for p in segment_pool.glob(f'seg-{DATASET_NAME}-{SCENE_ID:06d}-{SEGMENT_ID:06d}-*') if p.is_dir()
])
assert len(scene_candidates) > 0, '未找到 scene 资产目录'
assert len(seg_candidates) > 0, '未找到 segment 资产目录'

store = StreetForwardAssetStore(str(ASSET_ROOT), missing_policy='error')
scene_handle = store.get_scene_asset(DATASET_NAME, SCENE_ID)
scene_manifest = scene_handle.load_manifest()
seg_handle = store.verify_segment_asset(DATASET_NAME, SCENE_ID, SEGMENT_ID)
seg_manifest = seg_handle.load_manifest()

print('scene asset_id =', scene_manifest['asset_id'])
print('segment asset_id =', seg_manifest['asset_id'])

sample_refs = [(0, 0)]
meta_rows = scene_handle.load_image_meta(sample_refs)
print('image_table sample:', {
    'frame_idx': meta_rows[0]['frame_idx'],
    'cam_id': meta_rows[0]['cam_id'],
    'height': meta_rows[0]['height'],
    'width': meta_rows[0]['width'],
})

device = torch.device('cpu')
ds = build_multi_scene_dataset_v4(cfg, device=device)
ds.initialize()

print('training scenes:', ds.list_training_scene_ids())
print(f'segments in scene {SCENE_ID}:', ds.list_segment_ids(SCENE_ID))
print('segment index num_cams:', ds.get_segment_index(SCENE_ID, SEGMENT_ID).num_cams)


scene_pool exists   = True
segment_pool exists = True
registries exists   = True
scene asset_id = scene-nuscenes-000001-66304760
segment asset_id = seg-nuscenes-000001-000000-acc2c290
image_table sample: {'frame_idx': 0, 'cam_id': 0, 'height': 300, 'width': 533}
training scenes: [1]
segments in scene 1: [0]
segment index num_cams: 1


In [8]:
sidx = ds.get_segment_index(SCENE_ID, SEGMENT_ID)
assert len(sidx.train_image_refs) > 0, 'train_image_refs 为空，无法构造训练 batch'

source_ref = tuple(sidx.train_image_refs[0])
target_refs = [source_ref]
if len(sidx.train_image_refs) > 1:
    target_refs.append(tuple(sidx.train_image_refs[1]))

request = BatchRequestV4(
    scene_id=SCENE_ID,
    segment_id=SEGMENT_ID,
    source_image_ref=(int(source_ref[0]), int(source_ref[1])),
    target_image_refs=[(int(r[0]), int(r[1])) for r in target_refs],
    include_test=False,
)

batch = ds.get_segment_batch_from_image_refs(request, enforce_target0_equals_source=True)

# 集成测试式断言：关键 contract（不写死分辨率）
assert batch['source']['image'].shape[0] == 1
assert batch['target']['image'].shape[0] == len(target_refs)
assert batch['source']['viewdirs'].shape[-1] == 3
assert batch['source']['dynamic_mask'].ndim == 3
assert 'pointcloud' in batch
assert 'dynamic_info' in batch or len(batch['pointcloud'].get('dynamic', {})) == 0

print('batch keys:', sorted(batch.keys()))
print('source image shape:', tuple(batch['source']['image'].shape))
print('target image shape:', tuple(batch['target']['image'].shape))
print('source frame indices:', batch['source']['frame_indices'].tolist())
print('target frame indices:', batch['target']['frame_indices'].tolist())


batch keys: ['aabb', 'dynamic_info', 'pointcloud', 'request_meta', 'scene_folder_name', 'scene_id', 'segment_first_frame_idx', 'segment_first_pose', 'segment_first_pose_source', 'segment_id', 'source', 'target']
source image shape: (1, 300, 533, 3)
target image shape: (2, 300, 533, 3)
source frame indices: [0]
target frame indices: [0, 1]


/root/drivestudio-coding/datasets/multi_scene_dataset_v4.py:578: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at ../torch/csrc/utils/tensor_numpy.cpp:206.)
  mask = torch.as_tensor(arr, dtype=torch.float32)


In [9]:
# 演示 preload hint -> preload manager 的预热任务提交流程
hint = ds.build_preload_hint(
    scene_id=SCENE_ID,
    segment_id=SEGMENT_ID,
    future_image_refs=[(int(r[0]), int(r[1])) for r in target_refs],
    scope='next_block_exact',
)

ds.submit_preload_hint(
    hint=hint,
    hint_scope='next_block_exact',
    epoch_idx=0,
    global_step=0,
    block_idx_global=0,
    include_test=bool(OmegaConf.select(cfg, 'one_segment.include_test')),
)

for _ in range(5):
    time.sleep(0.05)

mgr = ds._preload_manager
if mgr is not None:
    stats = mgr.pop_stats()
    print('preload stats:', stats)
else:
    print('preload manager disabled')


preload manager disabled


In [11]:
scheduler = build_train_scheduler_v6_from_cfg(cfg, ds)

# toy trainer：用一个非常小的线性层去拟合 target 图像均值
model = torch.nn.Linear(3, 3, bias=False)
optimizer = torch.optim.SGD(model.parameters(), lr=1e-2)

loss_history = []
for step in range(4):
    train_batch = scheduler.next_batch()
    print('train_batch keys:', sorted(train_batch.keys()))



train_batch keys: ['_scheduler_v4_aligned_info', '_scheduler_v5_aligned_info', '_scheduler_v6_aligned_info', 'aabb', 'dynamic_info', 'pointcloud', 'request_meta', 'scene_folder_name', 'scene_id', 'segment_first_frame_idx', 'segment_first_pose', 'segment_first_pose_source', 'segment_id', 'source', 'target', 'test']
train_batch keys: ['_scheduler_v4_aligned_info', '_scheduler_v5_aligned_info', '_scheduler_v6_aligned_info', 'aabb', 'dynamic_info', 'pointcloud', 'request_meta', 'scene_folder_name', 'scene_id', 'segment_first_frame_idx', 'segment_first_pose', 'segment_first_pose_source', 'segment_id', 'source', 'target', 'test']
train_batch keys: ['_scheduler_v4_aligned_info', '_scheduler_v5_aligned_info', '_scheduler_v6_aligned_info', 'aabb', 'dynamic_info', 'pointcloud', 'request_meta', 'scene_folder_name', 'scene_id', 'segment_first_frame_idx', 'segment_first_pose', 'segment_first_pose_source', 'segment_id', 'source', 'target', 'test']
train_batch keys: ['_scheduler_v4_aligned_info', '_s

In [ ]:
events = scheduler.pop_events()
print("num events:", len(events))
print("event types:", [e.get("type") for e in events])
print("current info:", scheduler.get_current_info())


In [ ]:
# 清理后台线程（重复执行也安全）
ds.shutdown_preload()
print('cleanup done')
